# Coffee17 WR-HBP + Localized Top-k Self-Assessment Residual V1

Strict-deterministic five-fold development experiment. WR-HBP is trained with the frozen recipe, then frozen. A small top-5 localized reassessment residual is trained on existing mid-level spatial features. Outer test is not accessed.


In [ ]:
NOTEBOOK_BUILD = "WR-HBP-SAR-V1"
SCIENTIFIC_CODE_COMMIT = "5f2fbf3251b8bdf0379cc8f0a1f3dc932460f5d8"

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import importlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-wr-hbp-sar-code"
PROJECT = WORK / "coffee17-wr-hbp-self-assessment-project"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    env = os.environ.copy()
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True, env=env)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

prior_projects = sorted(
    p for p in INPUT.rglob("coffee17-wr-hbp-self-assessment-project")
    if p.is_dir()
)
if len(prior_projects) > 1:
    raise RuntimeError("Lebih dari satu prior SAR project ditemukan.")
if prior_projects:
    print("MERGE PRIOR SAR OUTPUT:", prior_projects[0])
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    shutil.copytree(prior_projects[0], PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", SCIENTIFIC_CODE_COMMIT])

requirements = REPO / "requirements/preprocessing-study.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")
print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)
print("GPU:", torch.cuda.get_device_name(0))

# Fail before expensive folds if the inherited WR path or SAR head is broken.
run([
    sys.executable, "-m", "pytest", "-q",
    "tests/modeling/test_wavelet_residual_hbp.py",
    "tests/modeling/test_self_assessment_residual.py",
], cwd=REPO)

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)
from bilinear_lmmd.core.config import load_config
from bilinear_lmmd.engine.physical_logit_residual_wr_hbp import configure_strict_determinism

CONFIG = REPO / "configs/wr_hbp_self_assessment/WR_HBP_SAR_V1.yaml"
OUT = PROJECT / "experiments/coffee17-wr-hbp-self-assessment-v1"
OUT.mkdir(parents=True, exist_ok=True)

det = configure_strict_determinism(42)
print("\n=== STRICT DETERMINISM ===")
print(json.dumps(det, indent=2))

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_wr_hbp_sar_original.zip"
PROV_LOCAL = WORK / "coffee17_wr_hbp_sar_provenance"
CANONICAL = WORK / "coffee17_wr_hbp_sar_original_v1"
FOLDS_LOCAL = WORK / "coffee17_wr_hbp_sar_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE,
    PROV_LOCAL,
    canonical_root=CANONICAL,
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

print("\n=== 5-FOLD WR-HBP BASE -> FROZEN SAR ===")
for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_wr_hbp_sar_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL / "clean_manifest.json",
        FOLDS_LOCAL / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_wr_hbp_self_assessment",
        "--config", CONFIG,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"WR_HBP_SAR_fold{fold}.log")

    result = json.loads(
        (OUT / f"fold_{fold}" / "seed42" / "pair_result.json").read_text()
    )
    d = result["DELTA_SAR_MINUS_BASE"]
    print(
        f"FOLD {fold} | "
        f"Base Macro={result['WR_HBP_BASE']['macro_f1']:.4f} | "
        f"SAR Macro={result['WR_HBP_SAR']['macro_f1']:.4f} | "
        f"Delta Macro={d['macro_f1']:+.4f} | "
        f"Delta Hard={d['hard_class_f1']:+.4f} | "
        f"Delta Worst={d['worst_class_f1']:+.4f} | "
        f"Rescue/Damage={result['paired_outcomes']['rescue']}/"
        f"{result['paired_outcomes']['damage']} | "
        f"AuditPairDelta={result['audited_pair_confusion_delta']['TOTAL']:+d}"
    )

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

SUMMARY = PROJECT / "analysis" / "wr_hbp_self_assessment_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_wr_hbp_self_assessment_summary",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
agg = summary["aggregate"]
d = agg["DELTA_SAR_MINUS_BASE"]

print("\n=== KEY RESULT ===")
print("Base Macro-F1 :", f"{agg['WR_HBP_BASE']['macro_f1']['mean']:.4%}")
print("SAR  Macro-F1 :", f"{agg['WR_HBP_SAR']['macro_f1']['mean']:.4%}")
print("Delta Macro   :", f"{d['macro_f1']['mean']:+.4%}")
print("Base Hard-F1  :", f"{agg['WR_HBP_BASE']['hard_class_f1']['mean']:.4%}")
print("SAR  Hard-F1  :", f"{agg['WR_HBP_SAR']['hard_class_f1']['mean']:.4%}")
print("Delta Hard    :", f"{d['hard_class_f1']['mean']:+.4%}")
print("Base Worst-F1 :", f"{agg['WR_HBP_BASE']['worst_class_f1']['mean']:.4%}")
print("SAR  Worst-F1 :", f"{agg['WR_HBP_SAR']['worst_class_f1']['mean']:.4%}")
print("Delta Worst   :", f"{d['worst_class_f1']['mean']:+.4%}")
print("Hard positive folds:", summary["positive_delta_folds"]["hard_class_f1"], "/ 5")
print("Rescue / damage:", summary["paired_prediction_outcomes"]["rescue"], "/",
      summary["paired_prediction_outcomes"]["damage"])
print("Audited pair delta:", summary["audited_pair_confusion_delta_total"]["TOTAL"])
print(
    "Base errors true-class in top5:",
    summary["base_error_top5_reachable"], "/",
    summary["base_error_count"],
    f"({summary['base_error_top5_reachable_fraction']:.2%})",
)
print("SCREENING GATE:", summary["screening_gate"]["decision"])

PKG = WORK / "wr-hbp-self-assessment-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)
shutil.copytree(PROJECT / "analysis", PKG / "analysis")

for fold in range(1, 6):
    fold_src = OUT / f"fold_{fold}" / "seed42"
    fold_dst = PKG / f"fold_{fold}"
    fold_dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(fold_src / "pair_result.json", fold_dst / "pair_result.json")

    for arm in ("WR_HBP_BASE", "WR_HBP_SAR"):
        src = fold_src / arm
        dst = fold_dst / arm
        dst.mkdir(parents=True, exist_ok=True)
        for name in ("run_contract.json", "history.json", "run_config.yaml"):
            p = src / name
            if p.is_file():
                shutil.copy2(p, dst / name)
        if (src / "validation").is_dir():
            shutil.copytree(src / "validation", dst / "validation")

shutil.copy2(CONFIG, PKG / "WR_HBP_SAR_V1.yaml")
for doc in (
    REPO / "docs/protocols/WR_HBP_SELF_ASSESSMENT_V1.md",
    REPO / "docs/analysis/PERSISTENT_ERROR_AUDIT_WR_HBP_V1.md",
    REPO / "docs/results/WR_HBP_GCE_V1_RESULTS.md",
):
    if doc.is_file():
        shutil.copy2(doc, PKG / doc.name)

zip_path = Path(shutil.make_archive(
    str(WORK / "wr-hbp-self-assessment-analysis-package"),
    "zip",
    root_dir=PKG,
))
print("\nREADY:", zip_path)
print("SIZE:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("Outer test was NOT accessed.")

for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
